#### Import necessary Python packages

In [1]:
import pytest
from pydantic import ValidationError
from fastapi import FastAPI, Depends
import json
import os
import sys
from pathlib import Path

#### Set the system path and read in the credentials

In [2]:
module_dir = os.path.abspath('../src/app')
sys.path.insert(0, module_dir)

In [3]:
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src" / "app").exists():
    ROOT = ROOT.parent

cred_path = (ROOT / "credentials" / "nwm-ciroh.json").resolve()
if not cred_path.exists():
    raise FileNotFoundError(f"Credentials file not found: {cred_path}")

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(cred_path)
sys.path.insert(0, str(ROOT / "src" / "app"))

#### Import validation models to test

In [4]:
from validator_models import GeometriesWithoutTimeParams, ReturnPeriodsParams, AnalysesAssimParams, ForecastsParams, RetrospectivesParams, FlowMetricsParams, FlowPercentilesParams, ReachesParams

/Applications/anaconda3/envs/nwmapi2/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


#### Define function to extract examples within the validation models

In [5]:
def extract_examples(valid_model):
    examples_data_dict = {}

    model_extra = valid_model.model_config.get("json_schema_extra", {}) or {}
    model_examples = model_extra.get("examples")
    if model_examples:
        examples_data_dict["__model__"] = model_examples if isinstance(model_examples, list) else [model_examples]

    def collect_examples(obj):
        if obj is None:
            return []

        values = []

        example = getattr(obj, "example", None)
        if example is not None:
            values.append(example)

        examples = getattr(obj, "examples", None)
        if examples:
            if isinstance(examples, dict):
                for item in examples.values():
                    values.append(item["value"] if isinstance(item, dict) and "value" in item else item)
            elif isinstance(examples, (list, tuple, set)):
                values.extend(examples)
            else:
                values.append(examples)

        extra = getattr(obj, "json_schema_extra", None) or {}
        if "example" in extra:
            values.append(extra["example"])
        if "examples" in extra:
            extra_examples = extra["examples"]
            values.extend(extra_examples if isinstance(extra_examples, list) else [extra_examples])

        return values

    for param_name, field in valid_model.model_fields.items():
        field_examples = collect_examples(field) + collect_examples(getattr(field, "default", None))
        if field_examples:
            deduped = []
            seen = set()
            for item in field_examples:
                key = json.dumps(item, sort_keys=True, default=str)
                if key not in seen:
                    seen.add(key)
                    deduped.append(item)
            examples_data_dict[param_name] = deduped

    return examples_data_dict

#### Test Geometries Endpoint Validation

Get the example parameters from GeometriesWithoutTimeParams model 

In [6]:
geometries_param_examples = extract_examples(GeometriesWithoutTimeParams)
geometries_param_examples

{'reach_id': ['1891586,2927567,3134443,3589508', '1891586'],
 'gage_id': ['13309220,13042500,13295000', '13309220'],
 'hydroshare_id': ['643dc03878704a30849536e302bdb2c0'],
 'bounding_box': ['-111.705,40.160,-111.582,40.331'],
 'geom_filter': ['POLYGON ((-111.93525458166658 40.40221765026246, -111.93525458166658 40.37069805865761, -111.88137384735077 40.37069805865761, -111.88137384735077 40.40221765026246, -111.93525458166658 40.40221765026246))',
  '{"type": "Polygon", "coordinates": [[[-111.93525458166658, 40.40221765026246], [-111.93525458166658, 40.37069805865761], [-111.88137384735077, 40.37069805865761], [-111.88137384735077, 40.40221765026246], [-111.93525458166658, 40.40221765026246]]]}',
  '010300000001000000050000004c6c0836dbfb5bc028e032de7b3344404c6c0836dbfb5bc00450b308732f4440f67ada6d68f85bc00450b308732f4440f67ada6d68f85bc028e032de7b3344404c6c0836dbfb5bc028e032de7b334440',
  'MULTIPOLYGON ( ((-111.93525458166658 40.40221765026246, -111.93525458166658 40.37069805865761, -11

Define the function to test bounding_box parameter within GeometriesWithoutTimeParams

In [7]:
def test_GeometriesWithoutTimeParams_bounding_box(params):
    try:
        geometryParams = GeometriesWithoutTimeParams(bounding_box=params)
        print(f"✅ Bounding box is valid: {geometryParams.bounding_box}")
    except ValidationError as e:
        for error in e.errors():
            loc = error['loc']
            if loc and loc[0] == '__root__':
                field = '__root__ (model-level validation)'
            else:
                field = loc[0] if loc else 'unknown'
            print(f"❌Error in field '{field}': {error['msg']} (type: {error['type']})")

Run the GeometriesWithoutTimeParams model for bounding_box test cases

In [8]:
print("Case 1: Valid bounding box from examples")
test_GeometriesWithoutTimeParams_bounding_box(geometries_param_examples.get("bounding_box", [])[0])
print("Case 2: Provide bounding box as a tuple instead of a string")
test_GeometriesWithoutTimeParams_bounding_box((-120.0, 30.0, -110.0, 40.0))
print("Case 3: Provide bounding box going outside valid ranges")
test_GeometriesWithoutTimeParams_bounding_box("-360.0, 30.0, -330.0, 40.0")
print("Case 4: Provide bounding box with too many coordinate values")
test_GeometriesWithoutTimeParams_bounding_box("-120.0, 30.0, -110.0, 40.0, 50.0")
print("Case 5: Provide bounding box with too few coordinate values")
test_GeometriesWithoutTimeParams_bounding_box("-120.0, 30.0, -110.0")
print("Case 6: Provide bounding box with non-numeric values")
test_GeometriesWithoutTimeParams_bounding_box("-120.0, thirty, -110.0, 40.0")
print("Case 7: Provide bounding box with min longitude greater than max longitude")
test_GeometriesWithoutTimeParams_bounding_box("-110.0, 30.0, -120.0, 40.0")
print("Case 8: Provide bounding box with None")
test_GeometriesWithoutTimeParams_bounding_box(None)

Case 1: Valid bounding box from examples
✅ Bounding box is valid: (-111.705, 40.16, -111.582, 40.331)
Case 2: Provide bounding box as a tuple instead of a string
❌Error in field 'bounding_box': Input should be a valid string (type: string_type)
Case 3: Provide bounding box going outside valid ranges
❌Error in field 'bounding_box': Value error, bounding_box min_lon coordinate (-360.0) out of the valid range (-180.0 to 180.0). (type: value_error)
Case 4: Provide bounding box with too many coordinate values
❌Error in field 'bounding_box': Value error, bounding_box doesn't match the required length of four comma-separated coordinates: min_lon,min_lat,max_lon,max_lat. (type: value_error)
Case 5: Provide bounding box with too few coordinate values
❌Error in field 'bounding_box': Value error, bounding_box doesn't match the required length of four comma-separated coordinates: min_lon,min_lat,max_lon,max_lat. (type: value_error)
Case 6: Provide bounding box with non-numeric values
❌Error in fie

Define the function to test geom_filter parameter within GeometriesWithoutTimeParams

In [9]:
def test_GeometriesWithoutTimeParams_geometry_filter(params):
    try:
        geometryParams = GeometriesWithoutTimeParams(geom_filter=params)
        print(f"✅ Geometry filter is valid: {geometryParams.geom_filter}")
    except ValidationError as e:
        for error in e.errors():
            loc = error['loc']
            if loc and loc[0] == '__root__':
                field = '__root__ (model-level validation)'
            else:
                field = loc[0] if loc else 'unknown'
            print(f"❌Error in field '{field}': {error['msg']} (type: {error['type']})")

Run the GeometriesWithoutTimeParams model for geom_filter test cases

In [10]:
print("Case 1: Valid WKT geometry filter from examples")
test_GeometriesWithoutTimeParams_geometry_filter(geometries_param_examples.get("geom_filter", [])[0])
print("Case 2: Valid GeoJSON geometry filter from examples")
test_GeometriesWithoutTimeParams_geometry_filter(geometries_param_examples.get("geom_filter", [])[1])
print("Case 3: Valid WKB geometry filter from examples")
test_GeometriesWithoutTimeParams_geometry_filter(geometries_param_examples.get("geom_filter", [])[2])
print("Case 4: Valid WKT MultiPolygon geometry filter from examples")
test_GeometriesWithoutTimeParams_geometry_filter(geometries_param_examples.get("geom_filter", [])[3])
print("Case 5: Valid GeoJSON MultiPolygon geometry filter from examples")
test_GeometriesWithoutTimeParams_geometry_filter(geometries_param_examples.get("geom_filter", [])[4])
print("Case 6: Valid WKB MultiPolygon geometry filter from examples")
test_GeometriesWithoutTimeParams_geometry_filter(geometries_param_examples.get("geom_filter", [])[5])
print("Case 7: Valid WKT MULTILINESTRING geometry filter from examples")
test_GeometriesWithoutTimeParams_geometry_filter(geometries_param_examples.get("geom_filter", [])[6])
print("Case 8: Valid WKT LINESTRING geometry filter from examples")
test_GeometriesWithoutTimeParams_geometry_filter(geometries_param_examples.get("geom_filter", [])[7])
print("Case 9: Valid WKT POINT geometry filter from examples")
test_GeometriesWithoutTimeParams_geometry_filter(geometries_param_examples.get("geom_filter", [])[8])
print("Case 10: Provide geometry filter with invalid WKT syntax")
test_GeometriesWithoutTimeParams_geometry_filter("LINESTRING (-111.93525458166658 40.40221765026246, -111.88137384735077 40.40221765026246, -111.88137384735077)")
print("Case 11: Provide geometry filter with invalid WKB syntax")
test_GeometriesWithoutTimeParams_geometry_filter("330300000001000000050000004c6c0836dbfb5bc028e032de7b3344404c6c0836dbfb5bc00450b308732f4440f67ada6d68f85bc00450b308732f4440f67ada6d68f85bc028e032de7b3344404c6c0836dbfb5bc028e032de7b334440")
print("Case 12: Provide geometry filter with invalid GeoJSON syntax")
test_GeometriesWithoutTimeParams_geometry_filter('{"type": "Poligon", "coordinates": [[[-111.93525458166658, 40.40221765026246], [-111.93525458166658, 40.37069805865761], [-111.88137384735077, 40.37069805865761], [-111.88137384735077, 40.40221765026246], [-111.93525458166658, 40.40221765026246]]]}')
print("Case 13: Provide geometry filter with None")
test_GeometriesWithoutTimeParams_geometry_filter(None)

Case 1: Valid WKT geometry filter from examples
✅ Geometry filter is valid: ('POLYGON ((-111.93525458166658 40.40221765026246, -111.93525458166658 40.37069805865761, -111.88137384735077 40.37069805865761, -111.88137384735077 40.40221765026246, -111.93525458166658 40.40221765026246))', 'wkt')
Case 2: Valid GeoJSON geometry filter from examples
✅ Geometry filter is valid: ('{"type": "Polygon", "coordinates": [[[-111.93525458166658, 40.40221765026246], [-111.93525458166658, 40.37069805865761], [-111.88137384735077, 40.37069805865761], [-111.88137384735077, 40.40221765026246], [-111.93525458166658, 40.40221765026246]]]}', 'geojson')
Case 3: Valid WKB geometry filter from examples
✅ Geometry filter is valid: ('010300000001000000050000004c6c0836dbfb5bc028e032de7b3344404c6c0836dbfb5bc00450b308732f4440f67ada6d68f85bc00450b308732f4440f67ada6d68f85bc028e032de7b3344404c6c0836dbfb5bc028e032de7b334440', 'wkb')
Case 4: Valid WKT MultiPolygon geometry filter from examples
✅ Geometry filter is valid: 

Define the function to test reach_id parameter within GeometriesWithoutTimeParams

In [11]:
def test_GeometriesWithoutTimeParams_reach_id(params):
    try:
        geometryParams = GeometriesWithoutTimeParams(reach_id=params)
        print(f"✅ Reach ID is valid: {geometryParams.reach_id}")
    except ValidationError as e:
        for error in e.errors():
            loc = error['loc']
            if loc and loc[0] == '__root__':
                field = '__root__ (model-level validation)'
            else:
                field = loc[0] if loc else 'unknown'
            print(f"❌Error in field '{field}': {error['msg']} (type: {error['type']})")

Run the GeometriesWithoutTimeParams model for reach_id test cases

In [12]:
print("Case 1: Valid reach ID listing string from examples")
test_GeometriesWithoutTimeParams_reach_id(geometries_param_examples.get("reach_id", [])[0])
print("Case 2: Valid single reach ID string from examples")
test_GeometriesWithoutTimeParams_reach_id(geometries_param_examples.get("reach_id", [])[1])
print("Case 3: Provide reach ID with invalid format")
test_GeometriesWithoutTimeParams_reach_id("1891586x")
print("Case 4: Provide reach ID with empty entries in the list string")
test_GeometriesWithoutTimeParams_reach_id("1891586,2927567,,3134443,3589508")
print("Case 5: Provide reach ID with extra spaces in the list string")
test_GeometriesWithoutTimeParams_reach_id("1891586,2927567,            3134443,3589508")
print("Case 6: Provide reach ID with None")
test_GeometriesWithoutTimeParams_reach_id(None)

Case 1: Valid reach ID listing string from examples
✅ Reach ID is valid: ['1891586', '2927567', '3134443', '3589508']
Case 2: Valid single reach ID string from examples
✅ Reach ID is valid: ['1891586']
Case 3: Provide reach ID with invalid format
❌Error in field 'reach_id': Value error, reach_id 1 contains non-digit values. reach_id should be a string of digits only. (type: value_error)
Case 4: Provide reach ID with empty entries in the list string
❌Error in field 'reach_id': Value error, reach_id 1 contains non-digit values. reach_id should be a string of digits only. (type: value_error)
Case 5: Provide reach ID with extra spaces in the list string
✅ Reach ID is valid: ['1891586', '2927567', '3134443', '3589508']
Case 6: Provide reach ID with None
✅ Reach ID is valid: None


Define the function to test gage_id parameter within GeometriesWithoutTimeParams

In [13]:
def test_GeometriesWithoutTimeParams_gage_id(params):
    try:
        geometryParams = GeometriesWithoutTimeParams(gage_id=params)
        print(f"✅ Gage ID is valid: {geometryParams.gage_id}")
    except ValidationError as e:
        for error in e.errors():
            loc = error['loc']
            if loc and loc[0] == '__root__':
                field = '__root__ (model-level validation)'
            else:
                field = loc[0] if loc else 'unknown'
            print(f"❌Error in field '{field}': {error['msg']} (type: {error['type']})")

Run the GeometriesWithoutTimeParams model for gage_id test cases

In [14]:
print("Case 1: Valid gage ID listing string from examples")
test_GeometriesWithoutTimeParams_gage_id(geometries_param_examples.get("gage_id", [])[0])
print("Case 2: Valid single gage ID string from examples")
test_GeometriesWithoutTimeParams_gage_id(geometries_param_examples.get("gage_id", [])[1])
print("Case 3: Provide gage ID with invalid format")
test_GeometriesWithoutTimeParams_gage_id("13309220x")
print("Case 4: Provide gage ID with extra spaces in the list string")
test_GeometriesWithoutTimeParams_gage_id("13309220,13042500,13295000,    13309220")
print("Case 5: Provide gage ID with None")
test_GeometriesWithoutTimeParams_gage_id(None)

Case 1: Valid gage ID listing string from examples
✅ Gage ID is valid: ['13309220', '13042500', '13295000']
Case 2: Valid single gage ID string from examples
✅ Gage ID is valid: ['13309220']
Case 3: Provide gage ID with invalid format
❌Error in field 'gage_id': Value error, gage_id 1 contains non-digit values. gage_id should be a string of digits only. (type: value_error)
Case 4: Provide gage ID with extra spaces in the list string
✅ Gage ID is valid: ['13309220', '13042500', '13295000', '13309220']
Case 5: Provide gage ID with None
✅ Gage ID is valid: None


Define the function to test hydroshare_id parameter within GeometriesWithoutTimeParams

In [15]:
def test_GeometriesWithoutTimeParams_hydroshare_id(params):
    try:
        geometryParams = GeometriesWithoutTimeParams(hydroshare_id=params)
        print(f"✅ HydroShare ID is valid: {geometryParams.hydroshare_id}")
    except ValidationError as e:
        for error in e.errors():
            loc = error['loc']
            if loc and loc[0] == '__root__':
                field = '__root__ (model-level validation)'
            else:
                field = loc[0] if loc else 'unknown'
            print(f"❌Error in field '{field}': {error['msg']} (type: {error['type']})")

Run the GeometriesWithoutTimeParams model for hydroshare_id test cases

In [16]:
print("Case 1: Valid HydroShare ID string from examples")
test_GeometriesWithoutTimeParams_hydroshare_id(geometries_param_examples.get("hydroshare_id", [])[0])
print("Case 2: Provide HydroShare ID with no valid resource to parse")
test_GeometriesWithoutTimeParams_hydroshare_id("9e4f3e42f75249e2ac5d413eeb80ebf4")
print("Case 3: Provide HydroShare ID with invalid format")
test_GeometriesWithoutTimeParams_hydroshare_id("9e4z3e42f75249e2ac5d413eeb80ebf4")
print("Case 4: Provide HydroShare ID with None")
test_GeometriesWithoutTimeParams_hydroshare_id(None)

Case 1: Valid HydroShare ID string from examples
✅ HydroShare ID is valid: https://www.hydroshare.org/resource/643dc03878704a30849536e302bdb2c0/data/contents/nwm_comids.json
Case 2: Provide HydroShare ID with no valid resource to parse
❌Error in field 'hydroshare_id': Value error, hydroshare_id provided is valid but no acceptable file found in the resource. Please make sure the resource contains a file named one of the following: ['nwm_comids.json', 'nwm_reach_ids.json', 'nwm_reachids.json', 'nwm_comid_list.json', 'nwm_comids_list.json', 'comids.json', 'comids_list.json', 'nwm_reaches.json', 'nwm_reachid_list.json', 'nwm_reachids_list.json', 'reaches.json', 'reach_id_list.json', 'reach_ids_list.json']. (type: value_error)
Case 3: Provide HydroShare ID with invalid format
❌Error in field 'hydroshare_id': Value error, hydroshare_id provided is not a 32-character hexadecimal string. (type: value_error)
Case 4: Provide HydroShare ID with None
✅ HydroShare ID is valid: None


Define the function to test huc parameter within GeometriesWithoutTimeParams

In [17]:
def test_GeometriesWithoutTimeParams_huc(params):
    try:
        geometryParams = GeometriesWithoutTimeParams(huc=params)
        print(f"✅ HUC is valid: {geometryParams.huc}")
    except ValidationError as e:
        for error in e.errors():
            print(f"❌Error in field '{error['loc'][0]}': {error['msg']} (type: {error['type']})")

Run the GeometriesWithoutTimeParams model for huc test cases

In [18]:
print("Case 1: Valid HUC string from examples")
test_GeometriesWithoutTimeParams_huc(geometries_param_examples.get("huc", [])[0])
print("Case 2: Valid HUC string from examples")
test_GeometriesWithoutTimeParams_huc(geometries_param_examples.get("huc", [])[1])
print("Case 3: Valid HUC string from examples")
test_GeometriesWithoutTimeParams_huc(geometries_param_examples.get("huc", [])[2])
print("Case 4: Valid HUC string from examples")
test_GeometriesWithoutTimeParams_huc(geometries_param_examples.get("huc", [])[3])
print("Case 5: Valid HUC string from examples")
test_GeometriesWithoutTimeParams_huc(geometries_param_examples.get("huc", [])[4])
print("Case 6: Valid HUC string from examples")
test_GeometriesWithoutTimeParams_huc(geometries_param_examples.get("huc", [])[5])
print("Case 7: Provide HUC with invalid (non-numeric: mix alphabet O for 0) format")
test_GeometriesWithoutTimeParams_huc("O2")
print("Case 8: Provide HUC with invalid (non-numeric) format")
test_GeometriesWithoutTimeParams_huc("huc16")
print("Case 9: Provide HUC in invalid format")
test_GeometriesWithoutTimeParams_huc("163")

Case 1: Valid HUC string from examples
✅ HUC is valid: 160202010500
Case 2: Valid HUC string from examples
✅ HUC is valid: 1602020105
Case 3: Valid HUC string from examples
✅ HUC is valid: 16020201
Case 4: Valid HUC string from examples
✅ HUC is valid: 160202
Case 5: Valid HUC string from examples
✅ HUC is valid: 1602
Case 6: Valid HUC string from examples
✅ HUC is valid: 16
Case 7: Provide HUC with invalid (non-numeric: mix alphabet O for 0) format
❌Error in field 'huc': Value error, huc doesn't match the string of prespecified digits. Find more info about valid HUC levels here: https://water.usgs.gov/themes/hydrologic-units/ (type: value_error)
Case 8: Provide HUC with invalid (non-numeric) format
❌Error in field 'huc': Value error, huc doesn't match the string of prespecified digits. Find more info about valid HUC levels here: https://water.usgs.gov/themes/hydrologic-units/ (type: value_error)
Case 9: Provide HUC in invalid format
❌Error in field 'huc': Value error, huc doesn't matc

Define the function to test lat and lon parameters within GeometriesWithoutTimeParams

In [19]:
def test_GeometriesWithoutTimeParams_latlon(params):
    try:
        geometryParams = GeometriesWithoutTimeParams(lat=params[0], lon=params[1])
        print(f"✅ Latitude is valid: {geometryParams.lat} + ✅ Longitude is valid: {geometryParams.lon}")
    except ValidationError as e:
        for error in e.errors():
            loc = error['loc']
            if loc and loc[0] == '__root__':
                field = '__root__ (model-level validation)'
            else:
                field = loc[0] if loc else 'unknown'
            print(f"❌Error in field '{field}': {error['msg']} (type: {error['type']})")

Run the GeometriesWithoutTimeParams model for lat and lon test cases

In [20]:
print("Case 1: Valid latitude and longitude from examples")
test_GeometriesWithoutTimeParams_latlon((geometries_param_examples.get("lat", [])[0], geometries_param_examples.get("lon", [])[0]))
print("Case 2: Valid latitude and longitude from examples")
test_GeometriesWithoutTimeParams_latlon((geometries_param_examples.get("lat", [])[1], geometries_param_examples.get("lon", [])[1]))
print("Case 3: Valid latitude and longitude from examples")
test_GeometriesWithoutTimeParams_latlon((geometries_param_examples.get("lat", [])[2], geometries_param_examples.get("lon", [])[2]))
print("Case 4: Provide latitude and longitude mixing up the order: lat for longitude and lon for latitude")
test_GeometriesWithoutTimeParams_latlon((geometries_param_examples.get("lon", [])[0], geometries_param_examples.get("lat", [])[0]))
print("Case 5: Provide longitude only with None for latitude")
test_GeometriesWithoutTimeParams_latlon((None, geometries_param_examples.get("lon", [])[1]))

Case 1: Valid latitude and longitude from examples
✅ Latitude is valid: 40.176187 + ✅ Longitude is valid: -111.786835
Case 2: Valid latitude and longitude from examples
✅ Latitude is valid: 47.576526 + ✅ Longitude is valid: -123.837891
Case 3: Valid latitude and longitude from examples
✅ Latitude is valid: 35.871247 + ✅ Longitude is valid: -76.234131
Case 4: Provide latitude and longitude mixing up the order: lat for longitude and lon for latitude
❌Error in field 'lat': Value error, 'lat' goes beyond the expected range of -90.0 to 90.0. (type: value_error)
Case 5: Provide longitude only with None for latitude
❌Error in field 'unknown': Value error, Both 'lon' and 'lat' must be provided together. Please provide both parameters or neither of them to switch to other filtering. (type: value_error)


Define the function to test output_format parameter within GeometriesWithoutTimeParams

In [21]:
def test_GeometriesWithoutTimeParams_output_format(params):
    try:
        geometryParams = GeometriesWithoutTimeParams(output_format=params, reach_id = '101')
        print(f"✅ Output format is valid: {geometryParams.output_format}")
    except ValidationError as e:
        for error in e.errors():
            loc = error['loc']
            if loc and loc[0] == '__root__':
                field = '__root__ (model-level validation)'
            else:
                field = loc[0] if loc else 'unknown'
            print(f"❌Error in field '{field}': {error['msg']} (type: {error['type']})")

Run the GeometriesWithoutTimeParams model for output_format test cases

In [24]:
print("Case 1: Valid output format from examples")
test_GeometriesWithoutTimeParams_output_format(geometries_param_examples.get("output_format", [])[0])
print("Case 2: Valid output format from examples")
test_GeometriesWithoutTimeParams_output_format("json")
print("Case 3: Provide output format not available for the endpoint")
test_GeometriesWithoutTimeParams_output_format("gpkg")
print("Case 4: Provide output format as supported alias")
test_GeometriesWithoutTimeParams_output_format("shp")
print("Case 5: Provide output format as unsupported alias")
test_GeometriesWithoutTimeParams_output_format("shapa")

Case 1: Valid output format from examples
✅ Output format is valid: geojson
Case 2: Valid output format from examples
✅ Output format is valid: json
Case 3: Provide output format not available for the endpoint
❌Error in field 'output_format': Value error, output_format is not one of [['csv', 'geojson', 'json', 'shapefile', 'shp']] for this endpoint. (type: value_error)
Case 4: Provide output format as supported alias
✅ Output format is valid: shp
Case 5: Provide output format as unsupported alias
❌Error in field 'output_format': Value error, output_format is not one of [['csv', 'geojson', 'json', 'shapefile', 'shp']] for this endpoint. (type: value_error)


Define the function to test lowest_stream_order parameter within GeometriesWithoutTimeParams

In [37]:
def test_GeometriesWithoutTimeParams_lowest_stream_order(params):
    try:
        geometryParams = GeometriesWithoutTimeParams(lowest_stream_order=params[0], huc=params[1])
        print(f"✅ Lowest stream order is valid: {geometryParams.lowest_stream_order}")
    except ValidationError as e:
        for error in e.errors():
            loc = error['loc']
            if loc and loc[0] == '__root__':
                field = '__root__ (model-level validation)'
            else:
                field = loc[0] if loc else 'unknown'
            print(f"❌Error in field '{field}': {error['msg']} (type: {error['type']})")

Run the GeometriesWithoutTimeParams model for lowest_stream_order test cases

In [39]:
print("Case 1: Valid lowest stream order and HUC from examples")
test_GeometriesWithoutTimeParams_lowest_stream_order([geometries_param_examples.get("lowest_stream_order", [])[0], "16020201"])
print("Case 2: Valid lowest stream order and HUC from examples")
test_GeometriesWithoutTimeParams_lowest_stream_order([geometries_param_examples.get("lowest_stream_order", [])[9], "16020201"])
print("Case 3: Provide lowest stream order with invalid (non-numeric) format")
test_GeometriesWithoutTimeParams_lowest_stream_order(["10", "16020201"])
print("Case 4: Provide lowest stream order in numeric format")
test_GeometriesWithoutTimeParams_lowest_stream_order([10, "16020201"])
print("Case 4: Provide lowest stream order outside valid range")
test_GeometriesWithoutTimeParams_lowest_stream_order([11, "16020201"])
print("Case 5: Provide lowest stream order with no supporting geometry parameters")
test_GeometriesWithoutTimeParams_lowest_stream_order([geometries_param_examples.get("lowest_stream_order", [])[9], None])

Case 1: Valid lowest stream order and HUC from examples
✅ Lowest stream order is valid: 2
Case 2: Valid lowest stream order and HUC from examples
✅ Lowest stream order is valid: 10
Case 3: Provide lowest stream order with invalid (non-numeric) format
✅ Lowest stream order is valid: 10
Case 4: Provide lowest stream order in numeric format
✅ Lowest stream order is valid: 10
Case 4: Provide lowest stream order outside valid range
❌Error in field 'lowest_stream_order': Input should be 1, 2, 3, 4, 5, 6, 7, 8, 9 or 10 (type: enum)
Case 5: Provide lowest stream order with no supporting geometry parameters
❌Error in field 'unknown': Value error, 'lowest_stream_order' is only compatible with 'huc', 'geom_filter', or 'bounding_box' filtering parameters. (type: value_error)


Define the function to test with_buffer parameter within GeometriesWithoutTimeParams

In [40]:
def test_GeometriesWithoutTimeParams_with_buffer(params):
    try:
        geometryParams = GeometriesWithoutTimeParams(with_buffer=params[0], huc=params[1])
        print(f"✅ Buffer is valid: {geometryParams.with_buffer}")
    except ValidationError as e:
        for error in e.errors():
            loc = error['loc']
            if loc and loc[0] == '__root__':
                field = '__root__ (model-level validation)'
            else:
                field = loc[0] if loc else 'unknown'
            print(f"❌Error in field '{field}': {error['msg']} (type: {error['type']})")


Run the GeometriesWithoutTimeParams model for with_buffer test cases

In [41]:
print("Case 1: Valid buffer and HUC from examples")
test_GeometriesWithoutTimeParams_with_buffer([geometries_param_examples.get("with_buffer", [])[0], "16020201"])
print("Case 2: Provide with_buffer with no applicable geometry parameters")
test_GeometriesWithoutTimeParams_with_buffer([geometries_param_examples.get("with_buffer", [])[0], None])
print("Case 3: Provide with_buffer outside valid range")
test_GeometriesWithoutTimeParams_with_buffer([2000, "16020201"])

Case 1: Valid buffer and HUC from examples
✅ Buffer is valid: 100.0
Case 2: Provide with_buffer with no applicable geometry parameters
❌Error in field 'unknown': Value error, 'with_buffer' is only applicable with 'geom_filter' or 'bounding_box' or 'huc' or 'lat' and 'lon' pair. (type: value_error)
Case 3: Provide with_buffer outside valid range
❌Error in field 'with_buffer': Value error, with_buffer must be between 0 and 1000 meters. (type: value_error)


Define the function to test ordered parameter within GeometriesWithoutTimeParams

In [42]:
def test_GeometriesWithoutTimeParams_ordered(params):
    try:
        geometryParams = GeometriesWithoutTimeParams(ordered=params, reach_id='101')
        print(f"✅ Ordered is valid: {geometryParams.ordered}")
    except ValidationError as e:
        for error in e.errors():
            loc = error['loc']
            if loc and loc[0] == '__root__':
                field = '__root__ (model-level validation)'
            else:
                field = loc[0] if loc else 'unknown'
            print(f"❌Error in field '{field}': {error['msg']} (type: {error['type']})")
  

Run the GeometriesWithoutTimeParams model for 'ordered' parameter test cases

In [43]:
print("Case 1: Valid ordered value from examples")
test_GeometriesWithoutTimeParams_ordered(geometries_param_examples.get("ordered", [])[0])
print("Case 2: Valid ordered value from examples")
test_GeometriesWithoutTimeParams_ordered(geometries_param_examples.get("ordered", [])[1])
print("Case 3: Provide an acceptable variant (Python boolean) of boolean true")
test_GeometriesWithoutTimeParams_ordered(True)
print("Case 4: Provide an acceptable variant (word string) of boolean true")
test_GeometriesWithoutTimeParams_ordered('true')
print("Case 5: Provide an acceptable variant (number string) of boolean true")
test_GeometriesWithoutTimeParams_ordered('1')
print("Case 6: Provide an acceptable variant (number string) of boolean false")
test_GeometriesWithoutTimeParams_ordered('0')
print("Case 7: Provide an acceptable variant (word string) of boolean false")
test_GeometriesWithoutTimeParams_ordered('false')
print("Case 8: Provide an acceptable variant (number) of boolean true")
test_GeometriesWithoutTimeParams_ordered(1)
print("Case 9: Provide an acceptable variant (number) of boolean false")
test_GeometriesWithoutTimeParams_ordered(0)
print("Case 10: Provide a number with no boolean meaning")
test_GeometriesWithoutTimeParams_ordered(-1)

Case 1: Valid ordered value from examples
✅ Ordered is valid: True
Case 2: Valid ordered value from examples
✅ Ordered is valid: False
Case 3: Provide an acceptable variant (Python boolean) of boolean true
✅ Ordered is valid: True
Case 4: Provide an acceptable variant (word string) of boolean true
✅ Ordered is valid: True
Case 5: Provide an acceptable variant (number string) of boolean true
✅ Ordered is valid: True
Case 6: Provide an acceptable variant (number string) of boolean false
✅ Ordered is valid: False
Case 7: Provide an acceptable variant (word string) of boolean false
✅ Ordered is valid: False
Case 8: Provide an acceptable variant (number) of boolean true
✅ Ordered is valid: True
Case 9: Provide an acceptable variant (number) of boolean false
✅ Ordered is valid: False
Case 10: Provide a number with no boolean meaning
❌Error in field 'ordered': Input should be a valid boolean, unable to interpret input (type: bool_parsing)


Define the function to test metadata parameter within GeometriesWithoutTimeParams

In [44]:
def test_GeometriesWithoutTimeParams_metadata(params):
    try:
        geometryParams = GeometriesWithoutTimeParams(metadata=params, reach_id = '101')
        print(f"✅ Metadata param is valid: {geometryParams.metadata}")
    except ValidationError as e:
        for error in e.errors():
            loc = error['loc']
            if loc and loc[0] == '__root__':
                field = '__root__ (model-level validation)'
            else:
                field = loc[0] if loc else 'unknown'
            print(f"❌Error in field '{field}': {error['msg']} (type: {error['type']})")
  

Run the GeometriesWithoutTimeParams model for metadata param test cases

In [45]:
print("Case 1: Valid metadata value from examples")
test_GeometriesWithoutTimeParams_metadata(geometries_param_examples.get("metadata", [])[0])
print("Case 2: Valid metadata value from examples")
test_GeometriesWithoutTimeParams_metadata(geometries_param_examples.get("metadata", [])[1])
print("Case 3: Provide an acceptable variant (Python boolean) of boolean true")
test_GeometriesWithoutTimeParams_metadata(True)
print("Case 4: Provide an acceptable variant (word string) of boolean true")
test_GeometriesWithoutTimeParams_metadata('true')
print("Case 5: Provide an acceptable variant (number string) of boolean true")
test_GeometriesWithoutTimeParams_metadata('1')
print("Case 6: Provide an acceptable variant (number string) of boolean false")
test_GeometriesWithoutTimeParams_metadata('0')
print("Case 7: Provide an acceptable variant (word string) of boolean false")
test_GeometriesWithoutTimeParams_metadata('false')
print("Case 8: Provide an acceptable variant (number) of boolean true")
test_GeometriesWithoutTimeParams_metadata(1)
print("Case 9: Provide an acceptable variant (number) of boolean false")
test_GeometriesWithoutTimeParams_metadata(0)
print("Case 10: Provide a number with no boolean meaning")
test_GeometriesWithoutTimeParams_metadata(-1)

Case 1: Valid metadata value from examples
✅ Metadata param is valid: True
Case 2: Valid metadata value from examples
✅ Metadata param is valid: False
Case 3: Provide an acceptable variant (Python boolean) of boolean true
✅ Metadata param is valid: True
Case 4: Provide an acceptable variant (word string) of boolean true
✅ Metadata param is valid: True
Case 5: Provide an acceptable variant (number string) of boolean true
✅ Metadata param is valid: True
Case 6: Provide an acceptable variant (number string) of boolean false
✅ Metadata param is valid: False
Case 7: Provide an acceptable variant (word string) of boolean false
✅ Metadata param is valid: False
Case 8: Provide an acceptable variant (number) of boolean true
✅ Metadata param is valid: True
Case 9: Provide an acceptable variant (number) of boolean false
✅ Metadata param is valid: False
Case 10: Provide a number with no boolean meaning
❌Error in field 'metadata': Input should be a valid boolean, unable to interpret input (type: bo

Define the function to test combination of parameters within GeometriesWithoutTimeParams

In [46]:
def test_GeometriesWithoutTimeParams(params):
    try:
        geometryParams = GeometriesWithoutTimeParams(**params)
        print(f"✅ Parameters are valid: {geometryParams}")
    except ValidationError as e:
        for error in e.errors():
            loc = error['loc']
            if loc and loc[0] == '__root__':
                field = '__root__ (model-level validation)'
            else:
                field = loc[0] if loc else 'unknown'
            print(f"❌Error in field '{field}': {error['msg']} (type: {error['type']})")
  

Run the GeometriesWithoutTimeParams model for comprehensive parameter test cases

In [50]:
print("Case 1: Provide all parameters from examples that are valid individually, but the combination is invalid due to model-level validation rules ")
test_GeometriesWithoutTimeParams(params = {'geom_filter': geometries_param_examples.get("metadata", [])[0],
                                           'ordered': geometries_param_examples.get("ordered", [])[0],
                                           'huc': geometries_param_examples.get("huc", [])[0],
                                           'lat': geometries_param_examples.get("lat", [])[0],
                                           'lon': geometries_param_examples.get("lon", [])[0],
                                           'reach_id': geometries_param_examples.get("reach_id", [])[0],
                                           'gage_id': geometries_param_examples.get("gage_id", [])[0],
                                           'hydroshare_id': geometries_param_examples.get("hydroshare_id", [])[0],
                                           'output_format': geometries_param_examples.get("output_format", [])[0],
                                           'lowest_stream_order': geometries_param_examples.get("lowest_stream_order", [])[0],
                                           'with_buffer': geometries_param_examples.get("with_buffer", [])[0]})
print("Case 2: Provide a valid combination of parameters from examples that satisfies model-level validation rules")
test_GeometriesWithoutTimeParams(params = {'geom_filter': geometries_param_examples.get("geom_filter", [])[0],
                                           'ordered': geometries_param_examples.get("ordered", [])[0],
                                           'output_format': geometries_param_examples.get("output_format", [])[0],
                                           'lowest_stream_order': geometries_param_examples.get("lowest_stream_order", [])[0],
                                           'with_buffer': geometries_param_examples.get("with_buffer", [])[0]})
print("Case 3: Provide only reach_id parameter")
test_GeometriesWithoutTimeParams(params = {'reach_id': geometries_param_examples.get("reach_id", [])[0],})
print("Case 4: Provide reach_id and with_buffer parameters together")
test_GeometriesWithoutTimeParams(params = {'reach_id': geometries_param_examples.get("reach_id", [])[0],
                                           'with_buffer': geometries_param_examples.get("with_buffer", [])[0]})
print("Case 5: Provide reach_id and bounding_box parameters together")
test_GeometriesWithoutTimeParams(params = {'reach_id': geometries_param_examples.get("reach_id", [])[0],
                                           'bounding_box': geometries_param_examples.get("bounding_box", [])[0],
                                           'with_buffer': geometries_param_examples.get("with_buffer", [])[0]})
print("Case 6: Provide bounding_box and with_buffer parameters together")
test_GeometriesWithoutTimeParams(params = {'bounding_box': geometries_param_examples.get("bounding_box", [])[0],
                                           'with_buffer': geometries_param_examples.get("with_buffer", [])[0]})
print("Case 7: Provide reach_id and gage_id parameters together")
test_GeometriesWithoutTimeParams(params = {'reach_id': geometries_param_examples.get("reach_id", [])[0],
                                           'gage_id': geometries_param_examples.get("gage_id", [])[0]})
print("Case 8: Provide gage_id and with_buffer parameters together")
test_GeometriesWithoutTimeParams(params = {'lat': geometries_param_examples.get("lat", [])[0],})
print("Case 9: Provide lat and lon parameters together with with_buffer")
test_GeometriesWithoutTimeParams(params = {'lat': geometries_param_examples.get("lat", [])[0],
                                           'lon': geometries_param_examples.get("lon", [])[0],
                                           'with_buffer': geometries_param_examples.get("with_buffer", [])[0]})
print("Case 10: Provide huc and with_buffer parameters together")
test_GeometriesWithoutTimeParams(params = {'huc': geometries_param_examples.get("huc", [])[-1],
                                           'with_buffer': geometries_param_examples.get("with_buffer", [])[1]})
print("Case 11: Provide geom_filter and with_buffer parameters together")
test_GeometriesWithoutTimeParams(params = {'geom_filter': geometries_param_examples.get("geom_filter", [])[-1],
                                           'with_buffer': geometries_param_examples.get("with_buffer", [])[1]})

Case 1: Provide all parameters from examples that are valid individually, but the combination is invalid due to model-level validation rules 
❌Error in field 'unknown': Value error, Several conflicting filtering parameters were provided. Please provide only one of the following filtering options: 'geom_filter', 'bounding_box', 'huc', 'reach_id', 'gage_id', 'hydroshare_id', or 'lon' and 'lat' pair. (type: value_error)
Case 2: Provide a valid combination of parameters from examples that satisfies model-level validation rules
✅ Parameters are valid: reach_id=Query(None) gage_id=Query(None) hydroshare_id=Query(None) bounding_box=Query(None) geom_filter=('POLYGON ((-111.93525458166658 40.40221765026246, -111.93525458166658 40.37069805865761, -111.88137384735077 40.37069805865761, -111.88137384735077 40.40221765026246, -111.93525458166658 40.40221765026246))', 'wkt') huc=Query(None) lon=Query(None) lat=Query(None) with_buffer=100.0 lowest_stream_order=2 output_format='geojson' ordered=True m

#### Test AnalysesAssimParams

Get the examples defined for various parameters in AnalysesAssimParams model

In [51]:
analysis_assim_param_examples = extract_examples(AnalysesAssimParams)
analysis_assim_param_examples

{'reach_id': ['1891586,2927567,3134443,3589508', '1891586'],
 'gage_id': ['13309220,13042500,13295000', '13309220'],
 'hydroshare_id': ['643dc03878704a30849536e302bdb2c0'],
 'bounding_box': ['-111.705,40.160,-111.582,40.331'],
 'geom_filter': ['POLYGON ((-111.93525458166658 40.40221765026246, -111.93525458166658 40.37069805865761, -111.88137384735077 40.37069805865761, -111.88137384735077 40.40221765026246, -111.93525458166658 40.40221765026246))',
  '{"type": "Polygon", "coordinates": [[[-111.93525458166658, 40.40221765026246], [-111.93525458166658, 40.37069805865761], [-111.88137384735077, 40.37069805865761], [-111.88137384735077, 40.40221765026246], [-111.93525458166658, 40.40221765026246]]]}',
  '010300000001000000050000004c6c0836dbfb5bc028e032de7b3344404c6c0836dbfb5bc00450b308732f4440f67ada6d68f85bc00450b308732f4440f67ada6d68f85bc028e032de7b3344404c6c0836dbfb5bc028e032de7b334440',
  'MULTIPOLYGON ( ((-111.93525458166658 40.40221765026246, -111.93525458166658 40.37069805865761, -11

Define the function to test time_zone parameter within AnalysesAssimParams model

In [52]:
def test_AnalysesAssimParams_time_zone(params):
    try:
        analysis_assim_param = AnalysesAssimParams(time_zone=params, reach_id = '101',
                                                  start_time=None,
                                                  end_time=None)
        print(f"✅ Timezone is valid: {analysis_assim_param.time_zone}")
    except ValidationError as e:
        for error in e.errors():
            loc = error['loc']
            if loc and loc[0] == '__root__':
                field = '__root__ (model-level validation)'
            else:
                field = loc[0] if loc else 'unknown'
            print(f"❌Error in field '{field}': {error['msg']} (type: {error['type']})")

Run the AnalysesAssimParams model for time_zone parameter test cases

In [53]:
print("Case 1: Valid timezone from examples")
test_AnalysesAssimParams_time_zone(analysis_assim_param_examples.get("time_zone", [])[0])
print("Case 2: Valid timezone from examples")
test_AnalysesAssimParams_time_zone(analysis_assim_param_examples.get("time_zone", [])[1])
print("Case 3: Valid timezone from examples")
test_AnalysesAssimParams_time_zone(analysis_assim_param_examples.get("time_zone", [])[2])
print("Case 4: Valid timezone from examples")
test_AnalysesAssimParams_time_zone(analysis_assim_param_examples.get("time_zone", [])[3])
print("Case 5: Provide a timezone in own words")
test_AnalysesAssimParams_time_zone('Utah Time')
print("Case 6: Provide a standard timezone")
test_AnalysesAssimParams_time_zone('MST')
print("Case 7: Provide a apparently standard timezone")
test_AnalysesAssimParams_time_zone('MT')
print("Case 8: Provide a apparently standard timezone")
test_AnalysesAssimParams_time_zone('MDT')

Case 1: Valid timezone from examples
✅ Timezone is valid: US/Mountain
Case 2: Valid timezone from examples
✅ Timezone is valid: UTC
Case 3: Valid timezone from examples
✅ Timezone is valid: America/Los_Angeles
Case 4: Valid timezone from examples
✅ Timezone is valid: US/Arizona
Case 5: Provide a timezone in own words
❌Error in field 'time_zone': Value error, 'Utah Time' is not a valid timezone. Please provide a valid timezone string (e.g. 'UTC', 'America/Los_Angeles', 'US/Mountain'). (type: value_error)
Case 6: Provide a standard timezone
✅ Timezone is valid: MST
Case 7: Provide a apparently standard timezone
❌Error in field 'time_zone': Value error, 'MT' is not a valid timezone. Please provide a valid timezone string (e.g. 'UTC', 'America/Los_Angeles', 'US/Mountain'). (type: value_error)
Case 8: Provide a apparently standard timezone
❌Error in field 'time_zone': Value error, 'MDT' is not a valid timezone. Please provide a valid timezone string (e.g. 'UTC', 'America/Los_Angeles', 'US/M

Define the function to test combination of parameters within AnalysesAssimParams model

In [55]:
def test_AnalysesAssimParams(params):
    try:
        analysis_assim_param = AnalysesAssimParams(**params)
        print(f"✅ AnalysesAssimParams are valid: {analysis_assim_param}")
    except ValidationError as e:
        for error in e.errors():
            loc = error['loc']
            if loc and loc[0] == '__root__':
                field = '__root__ (model-level validation)'
            else:
                field = loc[0] if loc else 'unknown'
            print(f"❌Error in field '{field}': {error['msg']} (type: {error['type']})")

Run the AnalysesAssimParams model for combined parameter test cases

In [ ]:

print("Case 1: Provide a valid combination of parameters from examples that satisfies model-level validation rules ")
test_AnalysesAssimParams(params = {'time_zone': 'US/Mountain',
                                   'start_time': analysis_assim_param_examples.get("start_time", [])[0],
                                   'end_time': analysis_assim_param_examples.get("end_time", [])[0],
                                  'huc':analysis_assim_param_examples.get("huc", [])[0]})
print("Case 2: Provide a combination of parameters, except a parameter identifying the reaches to look for")
test_AnalysesAssimParams(params = {'time_zone': 'UTC',
                                   'start_time': analysis_assim_param_examples.get("start_time", [])[0],
                                   'end_time': analysis_assim_param_examples.get("end_time", [])[0],})
print("Case 3: Provide a combination of parameters, leaving out the time parameters for defaults")
test_AnalysesAssimParams(params = {'time_zone': analysis_assim_param_examples.get("time_zone", [])[0],
                                  'start_time': None,
                                  'end_time':None,
                                  'huc':analysis_assim_param_examples.get("huc", [])[0]})
print("Case 4: Provide a combination of parameters with explicit mention of the run offsets to include")
test_AnalysesAssimParams(params = {'time_zone': analysis_assim_param_examples.get("time_zone", [])[0],
                                  'start_time': None,
                                  'end_time':None,
                                  'run_offset':"1,  2,  3",
                                  'huc':analysis_assim_param_examples.get("huc", [])[0]})
print("Case 5: Provide a combination of parameters with the run offsets going beyond expected values")
test_AnalysesAssimParams(params = {'time_zone': analysis_assim_param_examples.get("time_zone", [])[0],
                                  'start_time': None,
                                  'end_time':None,
                                  'run_offset':"1,  2,  3, 4, 5",
                                  'huc':analysis_assim_param_examples.get("huc", [])[0]})
print("Case 6: Provide the start and end times in the wrong order")
test_AnalysesAssimParams(params = {'time_zone': analysis_assim_param_examples.get("time_zone", [])[0],
                                  'start_time': analysis_assim_param_examples.get("end_time", [])[0],
                                  'end_time': analysis_assim_param_examples.get("start_time", [])[0],
                                  'huc':analysis_assim_param_examples.get("huc", [])[0]})
print("Case 7: Provide the start time going past the available data range")
test_AnalysesAssimParams(params = {'time_zone': analysis_assim_param_examples.get("time_zone", [])[0],
                                  'start_time': '2017-01-01T00:00:00',
                                  'end_time': analysis_assim_param_examples.get("end_time", [])[0],
                                  'huc':analysis_assim_param_examples.get("huc", [])[0]})
print("Case 8: Provide the start time just inside the available data range")
test_AnalysesAssimParams(params = {'time_zone': analysis_assim_param_examples.get("time_zone", [])[3],
                                  'start_time': '2018-09-16T22:00:00',
                                  'end_time': analysis_assim_param_examples.get("end_time", [])[0],
                                  'huc':analysis_assim_param_examples.get("huc", [])[0]})


Case 1: Provide a valid combination of parameters from examples that satisfies model-level validation rules 
✅ AnalysesAssimParams are valid: reach_id=Query(None) gage_id=Query(None) hydroshare_id=Query(None) bounding_box=Query(None) geom_filter=Query(None) huc='160202010500' lon=Query(None) lat=Query(None) with_buffer=Query(None) lowest_stream_order=Query(None) output_format=Query(json) ordered=Query(False) metadata=Query(False) time_zone='US/Mountain' start_time='2022-01-01 07:00:00 UTC' end_time='2022-02-01 07:00:00 UTC' run_offset=Query(1)
Case 2: Provide a combination of parameters, except a parameter identifying the reaches to look for
❌Error in field 'unknown': Value error, No reach filtering parameter provided. Please provide one of the following filtering options to specify the geometry or identifier to filter for: 'geom_filter', 'bounding_box', 'huc', 'reach_id', 'gage_id', 'hydroshare_id', or 'lon' and 'lat' pair. (type: value_error)
Case 3: Provide a combination of paramete

In [80]:
print("Case 9: Provide the start time apparently inside the available data range but missing it due to a different timezone")
test_AnalysesAssimParams(params = {'time_zone': 'Asia/Tokyo',
                                  'start_time': '2018-09-16T22:00:00',
                                  'end_time': analysis_assim_param_examples.get("end_time", [])[0],
                                  'huc':analysis_assim_param_examples.get("huc", [])[0]})
print("Case 10: Provide a different geometry filtering parameter")
test_AnalysesAssimParams(params = {'time_zone': 'UTC',
                                   'start_time': analysis_assim_param_examples.get("start_time", [])[1],
                                   'end_time': analysis_assim_param_examples.get("end_time", [])[1],
                                  'geom_filter':analysis_assim_param_examples.get("geom_filter", [])[0]})
print("Case 11: Provide a different geometry filtering parameter with the time parameters left out for defaults")
test_AnalysesAssimParams(params = {'time_zone': 'UTC',
                                   'start_time': analysis_assim_param_examples.get("start_time", [])[2],
                                   'end_time': analysis_assim_param_examples.get("end_time", [])[2],
                                  'bounding_box':analysis_assim_param_examples.get("bounding_box", [])[0]})
print("Case 12: Provide a different reach filtering parameter with the time parameters left out for defaults and the timezone set to UTC")
test_AnalysesAssimParams(params = {'time_zone': 'UTC',
                                   'start_time': analysis_assim_param_examples.get("start_time", [])[3],
                                   'end_time': analysis_assim_param_examples.get("end_time", [])[3],
                                  'gage_id':analysis_assim_param_examples.get("gage_id", [])[0]})
print("Case 13: Provide timezone aware start and end times.")
test_AnalysesAssimParams(params = {'time_zone': 'UTC',
                                   'start_time': "2022-01-01T00:00:00Z",
                                   'end_time': "2022-02-01T00:00:00Z",
                                  'huc':analysis_assim_param_examples.get("huc", [])[0]})
print("Case 14: Provide timezone aware start and end times in a different format.")
test_AnalysesAssimParams(params = {'time_zone': 'UTC',
                                   'start_time': "2022-01-01T00:00:00+06:00",
                                   'end_time': "2022-02-01T00:00:00+06:00",
                                  'huc':analysis_assim_param_examples.get("huc", [])[0]})
print("Case 15: Provide the end time going past the current time.")
test_AnalysesAssimParams(params = {'time_zone': 'UTC',
                                   'start_time': "2026-01-01T00:00:00",
                                   'end_time': "2027-02-01T00:00:00",
                                  'huc':analysis_assim_param_examples.get("huc", [])[0]})
print("Case 16: Provide a valid output format.")
test_AnalysesAssimParams(params = {'time_zone': 'UTC',
                                   'start_time': analysis_assim_param_examples.get("start_time", [])[0],
                                   'end_time': analysis_assim_param_examples.get("end_time", [])[0],
                                  'huc':analysis_assim_param_examples.get("huc", [])[0],
                                  'output_format': 'csv'})
print("Case 17: Provide a valid spatial output format.")
test_AnalysesAssimParams(params = {'time_zone': 'UTC',
                                   'start_time': analysis_assim_param_examples.get("start_time", [])[0],
                                   'end_time': analysis_assim_param_examples.get("end_time", [])[0],
                                  'huc':analysis_assim_param_examples.get("huc", [])[0],
                                  'output_format': 'gpkg'})
print("Case 18: Provide an invalid output format.")
test_AnalysesAssimParams(params = {'time_zone': 'UTC',
                                   'start_time': analysis_assim_param_examples.get("start_time", [])[0],
                                   'end_time': analysis_assim_param_examples.get("end_time", [])[0],
                                  'huc':analysis_assim_param_examples.get("huc", [])[0],
                                  'output_format': 'shapefile'})

Case 9: Provide the start time apparently inside the available data range but missing it due to a different timezone
❌Error in field 'unknown': Value error, start_time precedes the earliest available time for analyses-assim dataset: 2018-09-16T22:00:00 UTC. (type: value_error)
Case 10: Provide a different geometry filtering parameter
✅ AnalysesAssimParams are valid: reach_id=Query(None) gage_id=Query(None) hydroshare_id=Query(None) bounding_box=Query(None) geom_filter=('POLYGON ((-111.93525458166658 40.40221765026246, -111.93525458166658 40.37069805865761, -111.88137384735077 40.37069805865761, -111.88137384735077 40.40221765026246, -111.93525458166658 40.40221765026246))', 'wkt') huc=Query(None) lon=Query(None) lat=Query(None) with_buffer=Query(None) lowest_stream_order=Query(None) output_format=Query(json) ordered=Query(False) metadata=Query(False) time_zone='UTC' start_time='2022-01-01 00:00:00 UTC' end_time='2022-02-01 00:00:00 UTC' run_offset=Query(1)
Case 11: Provide a different 

#### Test RetrospectivesParams

Get the examples defined for various parameters in RetrospectivesParams model

In [67]:
retrospective_param_examples = extract_examples(RetrospectivesParams)
retrospective_param_examples

{'reach_id': ['1891586,2927567,3134443,3589508', '1891586'],
 'gage_id': ['13309220,13042500,13295000', '13309220'],
 'hydroshare_id': ['643dc03878704a30849536e302bdb2c0'],
 'bounding_box': ['-111.705,40.160,-111.582,40.331'],
 'geom_filter': ['POLYGON ((-111.93525458166658 40.40221765026246, -111.93525458166658 40.37069805865761, -111.88137384735077 40.37069805865761, -111.88137384735077 40.40221765026246, -111.93525458166658 40.40221765026246))',
  '{"type": "Polygon", "coordinates": [[[-111.93525458166658, 40.40221765026246], [-111.93525458166658, 40.37069805865761], [-111.88137384735077, 40.37069805865761], [-111.88137384735077, 40.40221765026246], [-111.93525458166658, 40.40221765026246]]]}',
  '010300000001000000050000004c6c0836dbfb5bc028e032de7b3344404c6c0836dbfb5bc00450b308732f4440f67ada6d68f85bc00450b308732f4440f67ada6d68f85bc028e032de7b3344404c6c0836dbfb5bc028e032de7b334440',
  'MULTIPOLYGON ( ((-111.93525458166658 40.40221765026246, -111.93525458166658 40.37069805865761, -11

Define the function to test combination of parameters within RetrospectivesParams model

In [68]:
def test_RetrospectivesParams(params):
    try:
        retrospectives_param = RetrospectivesParams(**params)
        print(f"✅ RetrospectivesParams are valid: {retrospectives_param}")
    except ValidationError as e:
        for error in e.errors():
            loc = error['loc']
            if loc and loc[0] == '__root__':
                field = '__root__ (model-level validation)'
            else:
                field = loc[0] if loc else 'unknown'
            print(f"❌Error in field '{field}': {error['msg']} (type: {error['type']})")

Run the RetrospectivesParams model for combined parameter test cases

In [79]:
print("Case 1: Provide a valid combination of parameters from examples that satisfies model-level validation rules ")
test_RetrospectivesParams(params = {'time_zone': retrospective_param_examples.get("time_zone", [])[0],
                                   'start_time': retrospective_param_examples.get("start_time", [])[0],
                                   'end_time': retrospective_param_examples.get("end_time", [])[0],
                                  'huc':retrospective_param_examples.get("huc", [])[0]})
print("Case 2: Provide a combination of parameters, except a parameter identifying the reaches to look for")
test_RetrospectivesParams(params = {'time_zone': retrospective_param_examples.get("time_zone", [])[1],
                                   'start_time': retrospective_param_examples.get("start_time", [])[0],
                                   'end_time': retrospective_param_examples.get("end_time", [])[0],})
print("Case 3: Provide a combination of parameters, leaving out the time parameters for default extraction")
test_RetrospectivesParams(params = {'time_zone': retrospective_param_examples.get("time_zone", [])[0],
                                  'start_time': None,
                                  'end_time':None,
                                  'huc':retrospective_param_examples.get("huc", [])[0]})
print("Case 4: Provide the start and end times in the wrong order")
test_RetrospectivesParams(params = {'time_zone': retrospective_param_examples.get("time_zone", [])[0],
                                  'start_time': retrospective_param_examples.get("end_time", [])[0],
                                  'end_time': retrospective_param_examples.get("start_time", [])[0],
                                  'huc':retrospective_param_examples.get("huc", [])[0]})
print("Case 5: Provide the start time going past the available data range")
test_RetrospectivesParams(params = {'time_zone': retrospective_param_examples.get("time_zone", [])[0],
                                  'start_time': '1970-01-01T00:00:00',
                                  'end_time': retrospective_param_examples.get("end_time", [])[0],
                                  'huc':retrospective_param_examples.get("huc", [])[0]})
print("Case 6: Provide the end time going beyond the the available data range")
test_RetrospectivesParams(params = {'time_zone': retrospective_param_examples.get("time_zone", [])[0],
                                  'start_time':retrospective_param_examples.get("end_time", [])[0],
                                  'end_time': '2026-01-01T00:00:00',
                                  'huc':retrospective_param_examples.get("huc", [])[0]})
print("Case 7: Provide a different geometry filtering parameter")
test_RetrospectivesParams(params = {'time_zone': retrospective_param_examples.get("time_zone", [])[0],
                                  'start_time': retrospective_param_examples.get("start_time", [])[1],
                                  'end_time': retrospective_param_examples.get("end_time", [])[1],
                                  'geom_filter':retrospective_param_examples.get("geom_filter", [])[0]})
print("Case 8: Provide a different geometry filtering parameter")
test_RetrospectivesParams(params = {'time_zone': retrospective_param_examples.get("time_zone", [])[1],
                                  'start_time': retrospective_param_examples.get("start_time", [])[2],
                                  'end_time': retrospective_param_examples.get("end_time", [])[2],
                                  'bounding_box':retrospective_param_examples.get("bounding_box", [])[0]})
print("Case 9: Provide a valid output format")
test_RetrospectivesParams(params = {'time_zone': 'UTC',
                                   'start_time': retrospective_param_examples.get("start_time", [])[0],
                                   'end_time': retrospective_param_examples.get("end_time", [])[0],
                                  'huc':retrospective_param_examples.get("huc", [])[0],
                                  'output_format': 'csv'})
print("Case 10: Provide a valid spatial output format")
test_RetrospectivesParams(params = {'time_zone': 'UTC',
                                   'start_time': retrospective_param_examples.get("start_time", [])[0],
                                   'end_time': retrospective_param_examples.get("end_time", [])[0],
                                  'huc':retrospective_param_examples.get("huc", [])[0],
                                  'output_format': 'geopackage'})
print("Case 11: Provide an invalid spatial output format")
test_RetrospectivesParams(params = {'time_zone': 'UTC',
                                   'start_time': retrospective_param_examples.get("start_time", [])[0],
                                   'end_time': retrospective_param_examples.get("end_time", [])[0],
                                  'huc':retrospective_param_examples.get("huc", [])[0],
                                  'output_format': 'shapefile'})

Case 1: Provide a valid combination of parameters from examples that satisfies model-level validation rules 
✅ RetrospectivesParams are valid: reach_id=Query(None) gage_id=Query(None) hydroshare_id=Query(None) bounding_box=Query(None) geom_filter=Query(None) huc='160202010500' lon=Query(None) lat=Query(None) with_buffer=Query(None) lowest_stream_order=Query(None) output_format=Query(json) ordered=Query(False) metadata=Query(False) time_zone='US/Mountain' start_time='2020-01-01 07:00:00 UTC' end_time='2020-02-01 07:00:00 UTC'
Case 2: Provide a combination of parameters, except a parameter identifying the reaches to look for
❌Error in field 'unknown': Value error, No reach filtering parameter provided. Please provide one of the following filtering options to specify the geometry or identifier to filter for: 'geom_filter', 'bounding_box', 'huc', 'reach_id', 'gage_id', 'hydroshare_id', or 'lon' and 'lat' pair. (type: value_error)
Case 3: Provide a combination of parameters, leaving out the

#### Test ForecastsParams

Get the examples defined for various parameters in ForecastsParams model

In [84]:
forecasts_params_examples = extract_examples(ForecastsParams)
forecasts_params_examples

{'reach_id': ['1891586,2927567,3134443,3589508', '1891586'],
 'gage_id': ['13309220,13042500,13295000', '13309220'],
 'hydroshare_id': ['643dc03878704a30849536e302bdb2c0'],
 'bounding_box': ['-111.705,40.160,-111.582,40.331'],
 'geom_filter': ['POLYGON ((-111.93525458166658 40.40221765026246, -111.93525458166658 40.37069805865761, -111.88137384735077 40.37069805865761, -111.88137384735077 40.40221765026246, -111.93525458166658 40.40221765026246))',
  '{"type": "Polygon", "coordinates": [[[-111.93525458166658, 40.40221765026246], [-111.93525458166658, 40.37069805865761], [-111.88137384735077, 40.37069805865761], [-111.88137384735077, 40.40221765026246], [-111.93525458166658, 40.40221765026246]]]}',
  '010300000001000000050000004c6c0836dbfb5bc028e032de7b3344404c6c0836dbfb5bc00450b308732f4440f67ada6d68f85bc00450b308732f4440f67ada6d68f85bc028e032de7b3344404c6c0836dbfb5bc028e032de7b334440',
  'MULTIPOLYGON ( ((-111.93525458166658 40.40221765026246, -111.93525458166658 40.37069805865761, -11

Define the function to test combination of parameters within ForecastsParams model

In [85]:
def test_ForecastsParams(params):
    try:
        forecasts_param = ForecastsParams(**params)
        print(f"✅ ForecastsParams are valid: {forecasts_param}")
    except ValidationError as e:
        for error in e.errors():
            loc = error['loc']
            if loc and loc[0] == '__root__':
                field = '__root__ (model-level validation)'
            else:
                field = loc[0] if loc else 'unknown'
            print(f"❌Error in field '{field}': {error['msg']} (type: {error['type']})")


Run the ForecastsParams model for combined parameter test cases

In [91]:
print("Case 1: Provide a valid combination of parameters from examples that satisfies model-level validation rules ")
test_ForecastsParams({'forecast_type': 'short_range',
                      'reference_time': forecasts_params_examples.get("reference_time", [])[0],
                      'time_zone': forecasts_params_examples.get("time_zone", [])[0],
                      'reach_id': forecasts_params_examples.get("reach_id", [])[0],
                      'ensemble': '0'})
print("Case 2: Provide the ensemble parameter with multiple values outside the valid range for particular forecast type")
test_ForecastsParams({'forecast_type': 'short_range',
                      'reference_time': forecasts_params_examples.get("reference_time", [])[0],
                      'time_zone': forecasts_params_examples.get("time_zone", [])[0],
                      'reach_id': forecasts_params_examples.get("reach_id", [])[0],
                      'ensemble': '0,1, 2, 3,4,5'})
print("Case 3: Provide the ensemble parameter with values acceptable for the forecast type")
test_ForecastsParams({'forecast_type': 'medium_range',
                      'reference_time': forecasts_params_examples.get("reference_time", [])[0],
                      'time_zone': forecasts_params_examples.get("time_zone", [])[0],
                      'reach_id': forecasts_params_examples.get("reach_id", [])[0],
                      'ensemble': '0,1, 2, 3,4,5'})
print("Case 4: Provide a reference time going past the available data range")
test_ForecastsParams({'forecast_type': 'medium_range',
                      'reference_time': '2017-01-01T00:00:00',
                      'time_zone': forecasts_params_examples.get("time_zone", [])[0],
                      'reach_id': forecasts_params_examples.get("reach_id", [])[0],
                      'ensemble': '0,1, 2, 3,4,5'})
print("Case 4: Provide a made-up forecast type")
test_ForecastsParams({'forecast_type': 'long_term',
                      'reference_time': '2017-01-01T00:00:00',
                      'time_zone': forecasts_params_examples.get("time_zone", [])[0],
                      'reach_id': forecasts_params_examples.get("reach_id", [])[0],
                      'ensemble': '0,1, 2, 3,4,5'})
print("Case 5: Provide the parameters, but leave out the reference time for defaults (to the latest available)")
test_ForecastsParams({'forecast_type': 'short_range',
                      'time_zone': forecasts_params_examples.get("time_zone", [])[0],
                      'reference_time': None,
                      'reach_id': forecasts_params_examples.get("reach_id", [])[0],
                      'ensemble': '0'})
print("Case 6: Provide the reference time beyond the current time")
test_ForecastsParams({'forecast_type': 'long_range',
                      'time_zone': forecasts_params_examples.get("time_zone", [])[0],
                      'reference_time': '2027-01-01T00:00:00',
                      'reach_id': forecasts_params_examples.get("reach_id", [])[0],
                      'ensemble': '0,1'})
print("Case 7: Provide the reference time past the available data range")
test_ForecastsParams({'forecast_type': 'long_range',
                      'time_zone': forecasts_params_examples.get("time_zone", [])[0],
                      'reference_time': '2017-01-01T00:00:00',
                      'reach_id': forecasts_params_examples.get("reach_id", [])[0],
                      'ensemble': '0,1'})
print("Provide a recent reference time for which the data is not available yet")
test_ForecastsParams({'forecast_type': 'short_range',
                      'time_zone': forecasts_params_examples.get("time_zone", [])[0],
                      'reference_time': '2026-03-21T00:00:00',
                      'reach_id': forecasts_params_examples.get("reach_id", [])[0],
                      'ensemble': '0'})
print("Case 8: Provide the ensemble parameter with values going beyond the valid range for the forecast type")
test_ForecastsParams({'forecast_type': 'long_range',
                      'reference_time': forecasts_params_examples.get("reference_time", [])[1],
                      'time_zone': forecasts_params_examples.get("time_zone", [])[0],
                      'reach_id': forecasts_params_examples.get("reach_id", [])[0],
                      'ensemble': '0,1, 2, 3,4,5,6'})
print("Case 9: Provide a valid output format")
test_ForecastsParams({'forecast_type': 'short_range',
                      'reference_time': forecasts_params_examples.get("reference_time", [])[1],
                      'time_zone': forecasts_params_examples.get("time_zone", [])[0],
                      'reach_id': forecasts_params_examples.get("reach_id", [])[0],
                      'ensemble': '0',
                      'output_format': 'csv'})
print("Case 10: Provide a valid spatial output format")
test_ForecastsParams({'forecast_type': 'short_range',
                      'reference_time': forecasts_params_examples.get("reference_time", [])[1],
                      'time_zone': forecasts_params_examples.get("time_zone", [])[0],
                      'reach_id': forecasts_params_examples.get("reach_id", [])[0],
                      'ensemble': '0',
                      'output_format': 'gpkg'})
print("Case 11: Provide an invalid spatial output format")
test_ForecastsParams({'forecast_type': 'short_range',
                      'reference_time': forecasts_params_examples.get("reference_time", [])[1],
                      'time_zone': forecasts_params_examples.get("time_zone", [])[0],
                      'reach_id': forecasts_params_examples.get("reach_id", [])[0],
                      'ensemble': '0',
                      'output_format': 'shapefile'})

Case 1: Provide a valid combination of parameters from examples that satisfies model-level validation rules 
✅ ForecastsParams are valid: reach_id=['1891586', '2927567', '3134443', '3589508'] gage_id=Query(None) hydroshare_id=Query(None) bounding_box=Query(None) geom_filter=Query(None) huc=Query(None) lon=Query(None) lat=Query(None) with_buffer=Query(None) lowest_stream_order=Query(None) output_format=Query(json) ordered=Query(False) metadata=Query(False) time_zone=<DstTzInfo 'US/Mountain' LMT-1 day, 17:00:00 STD> forecast_type=<ForecastOptions.short_range: 'short_range'> reference_time='2024-01-01 07:00:00 UTC' ensemble=[0]
Case 2: Provide the ensemble parameter with multiple values outside the valid range for particular forecast type
❌Error in field 'unknown': Value error, ensemble is not one of [0] for the ForecastOptions.short_range forecast_type. (type: value_error)
Case 3: Provide the ensemble parameter with values acceptable for the forecast type
✅ ForecastsParams are valid: rea

#### Test ReachesParams

Get the examples defined for various parameters in ReachesParams model

In [92]:
reaches_params_examples = extract_examples(ReachesParams)
reaches_params_examples

{'reach_id': [1129533, 1891586],
 'include': ['forecasts_short_range,flow_metrics',
  'forecasts_short_range,forecasts_medium_range,forecasts_long_range,analyses_assim,flow_metrics',
  'analyses_assim,return_periods'],
 'reference_time': ['2020-01-01T00:00:00',
  '2020/01/01 00:00:00',
  'Jan 1 2020 0:00 AM'],
 'time_zone': ['US/Mountain',
  'UTC',
  'America/Los_Angeles',
  'America/New_York'],
 'metadata': [True, False]}

Define the function to test combination of parameters within ReachesParams model

In [93]:
def test_ReachesParams(params):
    try:
        reaches_param = ReachesParams(**params)
        print(f"✅ ReachesParams are valid: {reaches_param}")
    except ValidationError as e:
        for error in e.errors():
            loc = error['loc']
            if loc and loc[0] == '__root__':
                field = '__root__ (model-level validation)'
            else:
                field = loc[0] if loc else 'unknown'
            print(f"❌Error in field '{field}': {error['msg']} (type: {error['type']})")

Run the ReachesParams model for combined parameter test cases

In [94]:
print("Case 1: Provide a reference time going past the available data range")
test_ReachesParams(params = {'reach_id': reaches_params_examples.get("reach_id", [])[0],
                             'reference_time': '2027-01-01T00:00:00',
                             'time_zone': 'UTC',
                             'include': None})
print("Case 2: Provide a very recent reference time for which the data is not available yet")
test_ReachesParams(params = {'reach_id': reaches_params_examples.get("reach_id", [])[0],
                             'reference_time': '2026-03-22T00:00:00',
                             'time_zone': 'UTC',
                             'include': None})
print("Case 3: Provide the parameters, but leave out the reference time for defaults (to the latest available)")
test_ReachesParams(params = {'reach_id': reaches_params_examples.get("reach_id", [])[0],
                             'reference_time': None,
                             'time_zone': 'UTC',
                             'include': None})
print("Case 4: Provide the parameters with a valid reference time but leave out the include parameter for defaults")
test_ReachesParams(params = {'reach_id': reaches_params_examples.get("reach_id", [])[0],
                             'reference_time': reaches_params_examples.get("reference_time", [])[0],
                             'time_zone': 'UTC',
                             'include': None})
print("Case 5: Provide the include parameter to include only intended datasets")
test_ReachesParams(params = {'reach_id': reaches_params_examples.get("reach_id", [])[0],
                             'reference_time': reaches_params_examples.get("reference_time", [])[0],
                             'time_zone': 'UTC',
                             'include': reaches_params_examples.get("include", [])[0]})
print("Case 5: Provide the include parameter with an invalid value")
test_ReachesParams(params = {'reach_id': reaches_params_examples.get("reach_id", [])[0],
                             'reference_time': reaches_params_examples.get("reference_time", [])[0],
                             'time_zone': 'UTC',
                             'include': "forecasts"})

Case 1: Provide a reference time going past the available data range
❌Error in field 'unknown': Value error, reference_time cannot be a future time. (type: value_error)
Case 2: Provide a very recent reference time for which the data is not available yet
❌Error in field 'unknown': Value error, reference_time is too recent and the forecast data for that reference time is not available yet. The latest available reference_time is 2026-01-25 23:00:00 UTC. (type: value_error)
Case 3: Provide the parameters, but leave out the reference time for defaults (to the latest available)
✅ ReachesParams are valid: reach_id=1129533 include=['forecasts_long_range', 'analyses_assim', 'percentile_flows', 'forecasts_short_range', 'flow_metrics', 'return_periods', 'forecasts_medium_range'] reference_time=('2026-01-25 23:00:00 UTC', '2026-01-25 18:00:00 UTC', '2026-01-25 18:00:00 UTC') time_zone='UTC' metadata=Query(False)
Case 4: Provide the parameters with a valid reference time but leave out the include p

#### Test FlowMetricsParams

Get the examples defined for various parameters in FlowMetricsParams model

In [97]:
flow_metrics_params_examples = extract_examples(FlowMetricsParams)
flow_metrics_params_examples

{'reach_id': ['1891586,2927567,3134443,3589508', '1891586'],
 'gage_id': ['13309220,13042500,13295000', '13309220'],
 'hydroshare_id': ['643dc03878704a30849536e302bdb2c0'],
 'bounding_box': ['-111.705,40.160,-111.582,40.331'],
 'geom_filter': ['POLYGON ((-111.93525458166658 40.40221765026246, -111.93525458166658 40.37069805865761, -111.88137384735077 40.37069805865761, -111.88137384735077 40.40221765026246, -111.93525458166658 40.40221765026246))',
  '{"type": "Polygon", "coordinates": [[[-111.93525458166658, 40.40221765026246], [-111.93525458166658, 40.37069805865761], [-111.88137384735077, 40.37069805865761], [-111.88137384735077, 40.40221765026246], [-111.93525458166658, 40.40221765026246]]]}',
  '010300000001000000050000004c6c0836dbfb5bc028e032de7b3344404c6c0836dbfb5bc00450b308732f4440f67ada6d68f85bc00450b308732f4440f67ada6d68f85bc028e032de7b3344404c6c0836dbfb5bc028e032de7b334440',
  'MULTIPOLYGON ( ((-111.93525458166658 40.40221765026246, -111.93525458166658 40.37069805865761, -11

Define the function to test combination of parameters within FlowMetricsParams model

In [98]:
def test_FlowMetricsParams(params):
    try:
        flow_metrics_param = FlowMetricsParams(**params)
        print(f"✅ FlowMetricsParams are valid: {flow_metrics_param}")
    except ValidationError as e:
        for error in e.errors():
            loc = error['loc']
            if loc and loc[0] == '__root__':
                field = '__root__ (model-level validation)'
            else:
                field = loc[0] if loc else 'unknown'
            print(f"❌Error in field '{field}': {error['msg']} (type: {error['type']})")

Run the FlowMetricsParams model for combined parameter test cases

In [101]:
print("Case 1: Provide parameters to get all metrics for a reach")
test_FlowMetricsParams(params = {'reach_id': flow_metrics_params_examples.get("reach_id", [])[0],
                                'metrics': None})
print("Case 2: Provide parameters to get specific metrics for reaches within a HUC")
test_FlowMetricsParams(params = {'huc': flow_metrics_params_examples.get("huc", [])[0],
                                'metrics': flow_metrics_params_examples.get("metrics", [])[0]})
print("Case 3: Provide the metrics parameter with an invalid metric name")
test_FlowMetricsParams(params = {'huc': flow_metrics_params_examples.get("huc", [])[0],
                                'metrics': "monthwise_mean, monthwise_max, monthwise_min"})
print("Case 4: Provide a valid output format")
test_FlowMetricsParams(params = {'huc': flow_metrics_params_examples.get("huc", [])[0],
                                'metrics': flow_metrics_params_examples.get("metrics", [])[1],
                                'output_format': 'csv'})
print("Case 5: Provide a valid spatial output format")
test_FlowMetricsParams(params = {'huc': flow_metrics_params_examples.get("huc", [])[0],
                                'metrics': flow_metrics_params_examples.get("metrics", [])[1],
                                'output_format': 'shapefile'})
print("Case 6: Provide an invalid spatial output format")
test_FlowMetricsParams(params = {'huc': flow_metrics_params_examples.get("huc", [])[0],
                                'metrics': flow_metrics_params_examples.get("metrics", [])[1],
                                'output_format': 'geopackage'})


Case 1: Provide parameters to get all metrics for a reach
✅ FlowMetricsParams are valid: reach_id=['1891586', '2927567', '3134443', '3589508'] gage_id=Query(None) hydroshare_id=Query(None) bounding_box=Query(None) geom_filter=Query(None) huc=Query(None) lon=Query(None) lat=Query(None) with_buffer=Query(None) lowest_stream_order=Query(None) output_format=Query(json) ordered=Query(False) metadata=Query(False) metrics='monthwise_mean, monthwise_cov, nth_percentile_flows, variability_index, slope_fdc, flashiness_index, sevenQ10, mean_annual_7_day_min, baseflow_index, zero_flow_days_n, low_flow_days_n, duration_low_flow_event, high_flow_days_n, duration_high_flow_event, half_flow_date, start_date_flood_season'
Case 2: Provide parameters to get specific metrics for reaches within a HUC
✅ FlowMetricsParams are valid: reach_id=Query(None) gage_id=Query(None) hydroshare_id=Query(None) bounding_box=Query(None) geom_filter=Query(None) huc='160202010500' lon=Query(None) lat=Query(None) with_buffer

#### Test FlowPercentilesParams

Get the examples defined for various parameters in FlowPercentilesParams model

In [102]:
flow_percentiles_params_examples = extract_examples(FlowPercentilesParams)
flow_percentiles_params_examples

{'reach_id': ['1891586,2927567,3134443,3589508', '1891586'],
 'gage_id': ['13309220,13042500,13295000', '13309220'],
 'hydroshare_id': ['643dc03878704a30849536e302bdb2c0'],
 'bounding_box': ['-111.705,40.160,-111.582,40.331'],
 'geom_filter': ['POLYGON ((-111.93525458166658 40.40221765026246, -111.93525458166658 40.37069805865761, -111.88137384735077 40.37069805865761, -111.88137384735077 40.40221765026246, -111.93525458166658 40.40221765026246))',
  '{"type": "Polygon", "coordinates": [[[-111.93525458166658, 40.40221765026246], [-111.93525458166658, 40.37069805865761], [-111.88137384735077, 40.37069805865761], [-111.88137384735077, 40.40221765026246], [-111.93525458166658, 40.40221765026246]]]}',
  '010300000001000000050000004c6c0836dbfb5bc028e032de7b3344404c6c0836dbfb5bc00450b308732f4440f67ada6d68f85bc00450b308732f4440f67ada6d68f85bc028e032de7b3344404c6c0836dbfb5bc028e032de7b334440',
  'MULTIPOLYGON ( ((-111.93525458166658 40.40221765026246, -111.93525458166658 40.37069805865761, -11

Define the function to test combination of parameters within FlowPercentilesParams model

In [103]:
def test_FlowPercentilesParams(params):
    try:
        flow_percentiles_param = FlowPercentilesParams(**params)
        print(f"✅ FlowPercentilesParams are valid: {flow_percentiles_param}")
    except ValidationError as e:
        for error in e.errors():
            loc = error['loc']
            if loc and loc[0] == '__root__':
                field = '__root__ (model-level validation)'
            else:
                field = loc[0] if loc else 'unknown'
            print(f"❌Error in field '{field}': {error['msg']} (type: {error['type']})")

Run the FlowPercentilesParams model for combined parameter test cases

In [104]:
print("Case 1: Provide parameters to get all percentiles for a reach")
test_FlowPercentilesParams(params = {'reach_id': flow_percentiles_params_examples.get("reach_id", [])[0],
                                'percentiles': None})
print("Case 2: Provide parameters to get specific percentiles for reaches within a HUC")
test_FlowPercentilesParams(params = {'huc': flow_percentiles_params_examples.get("huc", [])[0],
                                'percentiles': flow_percentiles_params_examples.get("percentiles", [])[0]})
print("Case 3: Provide the percentiles parameter with invalid percentile values")
test_FlowPercentilesParams(params = {'huc': flow_percentiles_params_examples.get("huc", [])[0],
                                'percentiles': "77, 88, 99"})
print("Case 4: Provide a valid output format")
test_FlowPercentilesParams(params = {'huc': flow_percentiles_params_examples.get("huc", [])[0],
                                'percentiles': flow_percentiles_params_examples.get("percentiles", [])[1],
                                'output_format': 'csv'})
print("Case 5: Provide a valid spatial output format")
test_FlowPercentilesParams(params = {'huc': flow_percentiles_params_examples.get("huc", [])[0],
                                'percentiles': flow_percentiles_params_examples.get("percentiles", [])[1],
                                'output_format': 'shapefile'})
print("Case 6: Provide an invalid spatial output format")
test_FlowPercentilesParams(params = {'huc': flow_percentiles_params_examples.get("huc", [])[0],
                                'percentiles': flow_percentiles_params_examples.get("percentiles", [])[1],
                                'output_format': 'geopackage'})

Case 1: Provide parameters to get all percentiles for a reach
✅ FlowPercentilesParams are valid: reach_id=['1891586', '2927567', '3134443', '3589508'] gage_id=Query(None) hydroshare_id=Query(None) bounding_box=Query(None) geom_filter=Query(None) huc=Query(None) lon=Query(None) lat=Query(None) with_buffer=Query(None) lowest_stream_order=Query(None) output_format=Query(json) ordered=Query(False) metadata=Query(False) percentiles=None
Case 2: Provide parameters to get specific percentiles for reaches within a HUC
✅ FlowPercentilesParams are valid: reach_id=Query(None) gage_id=Query(None) hydroshare_id=Query(None) bounding_box=Query(None) geom_filter=Query(None) huc='160202010500' lon=Query(None) lat=Query(None) with_buffer=Query(None) lowest_stream_order=Query(None) output_format=Query(json) ordered=Query(False) metadata=Query(False) percentiles=[0, 25, 50, 75, 100]
Case 3: Provide the percentiles parameter with invalid percentile values
❌Error in field 'percentiles': Value error, Invalid

#### Test ReturnPeriodsParams

Get the examples defined for various parameters in ReturnPeriodsParams model

In [105]:
return_periods_params_examples = extract_examples(ReturnPeriodsParams)
return_periods_params_examples

{'reach_id': ['1891586,2927567,3134443,3589508', '1891586'],
 'gage_id': ['13309220,13042500,13295000', '13309220'],
 'hydroshare_id': ['643dc03878704a30849536e302bdb2c0'],
 'bounding_box': ['-111.705,40.160,-111.582,40.331'],
 'geom_filter': ['POLYGON ((-111.93525458166658 40.40221765026246, -111.93525458166658 40.37069805865761, -111.88137384735077 40.37069805865761, -111.88137384735077 40.40221765026246, -111.93525458166658 40.40221765026246))',
  '{"type": "Polygon", "coordinates": [[[-111.93525458166658, 40.40221765026246], [-111.93525458166658, 40.37069805865761], [-111.88137384735077, 40.37069805865761], [-111.88137384735077, 40.40221765026246], [-111.93525458166658, 40.40221765026246]]]}',
  '010300000001000000050000004c6c0836dbfb5bc028e032de7b3344404c6c0836dbfb5bc00450b308732f4440f67ada6d68f85bc00450b308732f4440f67ada6d68f85bc028e032de7b3344404c6c0836dbfb5bc028e032de7b334440',
  'MULTIPOLYGON ( ((-111.93525458166658 40.40221765026246, -111.93525458166658 40.37069805865761, -11

Define the function to test combination of parameters within ReturnPeriodsParams model

In [106]:
def test_ReturnPeriodsParams(params):
    try:
        return_periods_param = ReturnPeriodsParams(**params)
        print(f"✅ ReturnPeriodsParams are valid: {return_periods_param}")
    except ValidationError as e:
        for error in e.errors():
            loc = error['loc']
            if loc and loc[0] == '__root__':
                field = '__root__ (model-level validation)'
            else:
                field = loc[0] if loc else 'unknown'
            print(f"❌Error in field '{field}': {error['msg']} (type: {error['type']})")

Run the ReturnPeriodsParams model for combined parameter test cases

In [107]:
print("Case 1: Provide parameters to get all return periods for a reach")
test_ReturnPeriodsParams(params = {'reach_id': return_periods_params_examples.get("reach_id", [])[0],
                                'return_periods': None})
print("Case 2: Provide parameters to get specific return periods for reaches within a HUC")
test_ReturnPeriodsParams(params = {'huc': return_periods_params_examples.get("huc", [])[0],
                                'return_periods': return_periods_params_examples.get("return_periods", [])[0]})
print("Case 3: Provide the return periods parameter with invalid values")
test_ReturnPeriodsParams(params = {'huc': return_periods_params_examples.get("huc", [])[0],
                                'return_periods': "77, 88, 99"})
print("Case 4: Provide a valid output format")
test_ReturnPeriodsParams(params = {'huc': return_periods_params_examples.get("huc", [])[0],
                                'return_periods': return_periods_params_examples.get("return_periods", [])[1],
                                'output_format': 'csv'})
print("Case 5: Provide a valid spatial output format")
test_ReturnPeriodsParams(params = {'huc': return_periods_params_examples.get("huc", [])[0],
                                'return_periods': return_periods_params_examples.get("return_periods", [])[1],
                                'output_format': 'shapefile'})
print("Case 6: Provide an invalid spatial output format")
test_ReturnPeriodsParams(params = {'huc': return_periods_params_examples.get("huc", [])[0],
                                'return_periods': return_periods_params_examples.get("return_periods", [])[1],
                                'output_format': 'geopackage'})

Case 1: Provide parameters to get all return periods for a reach
✅ ReturnPeriodsParams are valid: reach_id=['1891586', '2927567', '3134443', '3589508'] gage_id=Query(None) hydroshare_id=Query(None) bounding_box=Query(None) geom_filter=Query(None) huc=Query(None) lon=Query(None) lat=Query(None) with_buffer=Query(None) lowest_stream_order=Query(None) output_format=Query(json) ordered=Query(False) metadata=Query(False) return_periods=None
Case 2: Provide parameters to get specific return periods for reaches within a HUC
✅ ReturnPeriodsParams are valid: reach_id=Query(None) gage_id=Query(None) hydroshare_id=Query(None) bounding_box=Query(None) geom_filter=Query(None) huc='160202010500' lon=Query(None) lat=Query(None) with_buffer=Query(None) lowest_stream_order=Query(None) output_format=Query(json) ordered=Query(False) metadata=Query(False) return_periods='10,50,100'
Case 3: Provide the return periods parameter with invalid values
❌Error in field 'return_periods': Value error, return_period